In [1]:
from transformers import pipeline

#### 리뷰 긍/부정을 판단하기 위한 모델

In [2]:
model_negative_positive = 'WhitePeak/bert-base-cased-Korean-sentiment'
sentiment_model = pipeline(model=model_negative_positive)


reviews = [
    "이 제품은 배송도 빠르고 품질도 좋아서 정말 만족합니다.",
    "생각보다 성능이 훨씬 좋아서 다음에도 다시 구매하고 싶어요.",
    "디자인이 예쁘고 사용하기도 편해서 주변 사람들에게 추천하고 싶습니다.",
    "기대했던 것보다 품질이 너무 별로라서 실망했습니다.",
    "배송이 늦었고 제품에도 문제가 있어서 다시는 구매하고 싶지 않아요.",
    "사용하기 불편하고 가격에 비해 만족도가 많이 떨어집니다."
]

result = sentiment_model(reviews)

for review, result in zip(reviews, result):
    print(f"리뷰: {review}")
    print(f"감정 분석 결과: {result}\n")

pytorch_model.bin:   0%|          | 0.00/711M [00:00<?, ?B/s]

c:\Users\USER\anaconda3\envs\nlp_env\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--WhitePeak--bert-base-cased-Korean-sentiment. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/367 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/711M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

리뷰: 이 제품은 배송도 빠르고 품질도 좋아서 정말 만족합니다.
감정 분석 결과: {'label': 'LABEL_1', 'score': 0.9912688732147217}

리뷰: 생각보다 성능이 훨씬 좋아서 다음에도 다시 구매하고 싶어요.
감정 분석 결과: {'label': 'LABEL_1', 'score': 0.9892605543136597}

리뷰: 디자인이 예쁘고 사용하기도 편해서 주변 사람들에게 추천하고 싶습니다.
감정 분석 결과: {'label': 'LABEL_1', 'score': 0.9918220639228821}

리뷰: 기대했던 것보다 품질이 너무 별로라서 실망했습니다.
감정 분석 결과: {'label': 'LABEL_0', 'score': 0.9971703886985779}

리뷰: 배송이 늦었고 제품에도 문제가 있어서 다시는 구매하고 싶지 않아요.
감정 분석 결과: {'label': 'LABEL_0', 'score': 0.9960657954216003}

리뷰: 사용하기 불편하고 가격에 비해 만족도가 많이 떨어집니다.
감정 분석 결과: {'label': 'LABEL_0', 'score': 0.9970180988311768}



#### 신뢰성 판단하기 위한 모델
- 임베딩 하는 모델임

In [3]:
from transformers import AutoTokenizer, AutoModel
import torch
import torch.nn.functional as F

model_reliability = 'jhgan/ko-sbert-sts'
# reliability_model = pipeline(model=model_reliability)

tokenizer = AutoTokenizer.from_pretrained(model_reliability)
model = AutoModel.from_pretrained(model_reliability)

#Mean Pooling - Take attention mask into account for correct averaging
def mean_pooling(model_output, attention_mask):
    token_embeddings = model_output[0] #First element of model_output contains all token embeddings
    input_mask_expanded = attention_mask.unsqueeze(-1).expand(token_embeddings.size()).float()
    return torch.sum(token_embeddings * input_mask_expanded, 1) / torch.clamp(input_mask_expanded.sum(1), min=1e-9)


reviews = [
    "이 제품은 배송도 빠르고 품질도 좋아서 정말 만족합니다.",
    "생각보다 성능이 훨씬 좋아서 다음에도 다시 구매하고 싶어요.",
    "디자인이 예쁘고 사용하기도 편해서 주변 사람들에게 추천하고 싶습니다.",
    "가격 대비 기능이 다양하고 마감도 괜찮아서 만족스럽습니다.",
    "포장도 깔끔하고 실제로 사용해보니 기대 이상으로 편리했습니다.",
    "처음엔 걱정했는데 막상 써보니 성능이 좋아서 만족합니다.",

    "기대했던 것보다 품질이 너무 별로라서 실망했습니다.",
    "배송이 늦었고 제품에도 문제가 있어서 다시는 구매하고 싶지 않아요.",
    "사용하기 불편하고 가격에 비해 만족도가 많이 떨어집니다.",
    "설명과 실제 제품이 달라서 믿음이 가지 않았습니다.",
    "몇 번 사용하지 않았는데 벌써 고장이 나서 너무 아쉽습니다.",
    "디자인은 괜찮지만 성능이 기대 이하라서 추천하기 어렵습니다."
]

encoded_input = tokenizer(reviews, padding=True, truncation=True, return_tensors='pt')


with torch.no_grad():
    model_output = model(**encoded_input)

sentence_embeddings = mean_pooling(model_output, encoded_input['attention_mask'])



sentence_embeddings = F.normalize(sentence_embeddings, p=2, dim=1)

similarity = torch.matmul(sentence_embeddings, sentence_embeddings.T)

for i, review in enumerate(reviews):
    print(f"리뷰 {i+1}: {review}")
    print("다른 리뷰들과의 유사도:")
    
    for j, score in enumerate(similarity[i]):
        print(f"  리뷰 {j+1}와 유사도: {score.item():.4f}")
    
    print()


config.json:   0%|          | 0.00/620 [00:00<?, ?B/s]

c:\Users\USER\anaconda3\envs\nlp_env\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\USER\.cache\huggingface\hub\models--jhgan--ko-sbert-sts. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


tokenizer_config.json:   0%|          | 0.00/538 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/443M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: jhgan/ko-sbert-sts
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


리뷰 1: 이 제품은 배송도 빠르고 품질도 좋아서 정말 만족합니다.
다른 리뷰들과의 유사도:
  리뷰 1와 유사도: 1.0000
  리뷰 2와 유사도: 0.5637
  리뷰 3와 유사도: 0.5003
  리뷰 4와 유사도: 0.5916
  리뷰 5와 유사도: 0.6370
  리뷰 6와 유사도: 0.5464
  리뷰 7와 유사도: 0.4364
  리뷰 8와 유사도: 0.6356
  리뷰 9와 유사도: 0.3951
  리뷰 10와 유사도: 0.3867
  리뷰 11와 유사도: 0.3523
  리뷰 12와 유사도: 0.3129

리뷰 2: 생각보다 성능이 훨씬 좋아서 다음에도 다시 구매하고 싶어요.
다른 리뷰들과의 유사도:
  리뷰 1와 유사도: 0.5637
  리뷰 2와 유사도: 1.0000
  리뷰 3와 유사도: 0.5186
  리뷰 4와 유사도: 0.6274
  리뷰 5와 유사도: 0.5495
  리뷰 6와 유사도: 0.7489
  리뷰 7와 유사도: 0.4540
  리뷰 8와 유사도: 0.5435
  리뷰 9와 유사도: 0.4408
  리뷰 10와 유사도: 0.3651
  리뷰 11와 유사도: 0.4940
  리뷰 12와 유사도: 0.5279

리뷰 3: 디자인이 예쁘고 사용하기도 편해서 주변 사람들에게 추천하고 싶습니다.
다른 리뷰들과의 유사도:
  리뷰 1와 유사도: 0.5003
  리뷰 2와 유사도: 0.5186
  리뷰 3와 유사도: 1.0000
  리뷰 4와 유사도: 0.6596
  리뷰 5와 유사도: 0.6114
  리뷰 6와 유사도: 0.5492
  리뷰 7와 유사도: 0.2542
  리뷰 8와 유사도: 0.3225
  리뷰 9와 유사도: 0.4900
  리뷰 10와 유사도: 0.2714
  리뷰 11와 유사도: 0.4273
  리뷰 12와 유사도: 0.4519

리뷰 4: 가격 대비 기능이 다양하고 마감도 괜찮아서 만족스럽습니다.
다른 리뷰들과의 유사도:
  리뷰 1와 유사도: 0.5916
  리뷰 2와 유사도: 0.6274
  리뷰 3와 유사

model.safetensors:   0%|          | 0.00/442M [00:00<?, ?B/s]